In [ ]:
import cv2                      # OpenCV: xử lý ảnh (đọc, resize, biến đổi ảnh)
import numpy as np              # NumPy: xử lý mảng số, tensor
import pandas as pd             # Pandas: xử lý dữ liệu dạng bảng (CSV, DataFrame)
import matplotlib.pyplot as plt # Matplotlib: vẽ biểu đồ, hiển thị ảnh

from pathlib import Path        # Xử lý đường dẫn file/thư mục dạng đối tượng
import os.path                 # Làm việc với đường dẫn hệ thống (join, exists, ...)

from tensorflow.keras.preprocessing import image
# Các hàm load và tiền xử lý ảnh cho Keras

from tensorflow.keras.applications import MobileNet
# MobileNet pretrained (ImageNet) – backbone CNN nhẹ cho classification

from tensorflow.keras.layers import Dense
# Lớp fully-connected (output hoặc hidden layer)

from tensorflow.keras.preprocessing.image import ImageDataGenerator
# Tạo data generator + data augmentation cho tập ảnh

import tensorflow as tf
# Thư viện TensorFlow chính (train, model, GPU, backend)


In [5]:
import pandas as pd              # Thư viện xử lý dữ liệu dạng bảng (DataFrame)
import os                        # Thư viện làm việc với hệ thống file/thư mục

# Hàm lấy đường dẫn ảnh và nhãn tương ứng từ thư mục dataset
def get_paths_labels(root_dir):
    data = []                    # Danh sách lưu [đường_dẫn_ảnh, nhãn]
    
    for label in os.listdir(root_dir):          # Duyệt qua từng thư mục class
        class_dir = os.path.join(root_dir, label)
        
        if os.path.isdir(class_dir):             # Chỉ xử lý nếu là thư mục
            for img in os.listdir(class_dir):    # Duyệt từng file ảnh trong class
                if img.lower().endswith(('.jpg', '.png', '.jpeg')):  # Lọc file ảnh
                    data.append([
                        os.path.join(class_dir, img),  # Đường dẫn đầy đủ tới ảnh
                        label                            # Tên thư mục = nhãn
                    ])
    
    return pd.DataFrame(data, columns=['path', 'label'])  # Trả về DataFrame

# 🔹 Đường dẫn tới tập train đã được chia sẵn
train_dir = "/kaggle/input/d/ngobinhxuyen/asl-alphabet/asl_split/train"

# 🔹 Đường dẫn tới tập validation đã được chia sẵn
val_dir   = "/kaggle/input/d/ngobinhxuyen/asl-alphabet/asl_split/val"

# 🔹 Tạo DataFrame cho tập train (path + label)
train_data = get_paths_labels(train_dir)

# 🔹 Tạo DataFrame cho tập validation (path + label)
val_data  = get_paths_labels(val_dir)

# 🔹 Kiểm tra nhanh số lượng mẫu trong mỗi tập
print("Train:", train_data.shape)   # (số_ảnh_train, 2)
print("Val:", val_data.shape)       # (số_ảnh_val, 2)

# 🔹 Xem trước 5 dòng đầu của tập train
train_data.head()


Train: (55680, 2)
Val: (13920, 2)


,path,label
0,/kaggle/input/d/ngobinhxuyen/asl-alphabet/asl_...,N
1,/kaggle/input/d/ngobinhxuyen/asl-alphabet/asl_...,N
2,/kaggle/input/d/ngobinhxuyen/asl-alphabet/asl_...,N
3,/kaggle/input/d/ngobinhxuyen/asl-alphabet/asl_...,N
4,/kaggle/input/d/ngobinhxuyen/asl-alphabet/asl_...,N


In [6]:
os.makedirs("data_yolo/train/images", exist_ok=True)
os.makedirs("data_yolo/train/labels", exist_ok=True)
os.makedirs("data_yolo/val/images", exist_ok=True)
os.makedirs("data_yolo/val/labels", exist_ok=True)

In [8]:
classes = sorted(train_data['label'].unique())
class_to_id = {c: i for i, c in enumerate(classes)}

In [9]:
!ls -R data_yolo | head -50

data_yolo:
train
val

data_yolo/train:
images
labels

data_yolo/train/images:

data_yolo/train/labels:

data_yolo/val:
images
labels

data_yolo/val/images:

data_yolo/val/labels:


In [15]:
def create_yolo_files(df, split):
    # df   : DataFrame chứa cột 'path' (đường dẫn ảnh) và 'label' (tên lớp)
    # split: 'train' hoặc 'val' để lưu đúng thư mục YOLO

    for idx, row in df.iterrows():              # Duyệt từng dòng trong DataFrame
        img = cv2.imread(row['path'])           # Đọc ảnh từ đường dẫn
        
        if img is None:                         # Bỏ qua nếu ảnh lỗi/không đọc được
            continue
        
        h, w = img.shape[:2]                    # Lấy chiều cao và chiều rộng ảnh
        
        label_id = class_to_id[row['label']]    # Chuyển tên class → ID số (YOLO yêu cầu)

        # Bounding box ảo cho bài toán classification:
        # class_id x_center y_center width height (chuẩn YOLO, chuẩn hóa 0–1)
        # 0.5 0.5 1.0 1.0 = box bao phủ toàn bộ ảnh
        label_txt = f"{label_id} 0.5 0.5 1.0 1.0\n"
        
        save_name = os.path.basename(row['path'])  # Lấy tên file ảnh (không có path)
        
        # Đường dẫn lưu ảnh theo cấu trúc YOLO
        img_save_path = f"data_yolo/{split}/images/{save_name}"
        
        # Đường dẫn lưu file label (.txt) tương ứng với ảnh
        lbl_save_path = f"data_yolo/{split}/labels/{save_name.split('.')[0]}.txt"

        cv2.imwrite(img_save_path, img)          # Ghi ảnh sang thư mục YOLO
        
        with open(lbl_save_path, "w") as f:      # Ghi file nhãn YOLO
            f.write(label_txt)

# Tạo dữ liệu YOLO cho tập train
create_yolo_files(train_data, "train")

# Tạo dữ liệu YOLO cho tập validation
create_yolo_files(val_data, "val")


In [16]:
with open("data_yolo/data.yaml", "w") as f:
    f.write("train: train/images\n")
    f.write("val: val/images\n")
    f.write(f"nc: {len(classes)}\n")
    f.write("names: " + str(classes))

In [17]:
!pip install ultralytics --upgrade

In [18]:
!cat data_yolo/data.yaml
!ls data_yolo/train/images | head
!ls data_yolo/val/images | head


train: train/images
val: val/images
nc: 29
names: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']A1003.jpg
A1004.jpg
A1005.jpg
A1006.jpg
A100.jpg
A1010.jpg
A1012.jpg
A1013.jpg
A1015.jpg
A1016.jpg
ls: write error: Broken pipe
A1000.jpg
A1007.jpg
A1018.jpg
A1021.jpg
A1029.jpg
A1030.jpg
A1040.jpg
A1046.jpg
A1049.jpg
A104.jpg
ls: write error: Broken pipe


In [20]:
!pip install numpy==1.26.4 --force-reinstall


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 86.2 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.15.3 which is incompatible.
datasets 4.1.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires google-auth==2.38.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11m.pt")  

results = model.train(
    data="/kaggle/working/data_yolo/data.yaml",
    epochs=200,
    imgsz=224,
    batch= 512,
    patience=10,
    lr0=0.001,
    weight_decay=0.0005,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10, translate=0.1, scale=0.5, shear=0.2,
    perspective=0.0005,
    flipud=0.0, fliplr=0.5,
    project="/kaggle/working/runs",
    name="gesture_train_v1",
    exist_ok=True,
    device=[0,1]  
)



Ultralytics 8.3.228 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
                                                       CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=512, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_yolo/data.yaml, degrees=10, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=gesture_train_v1, nbs=64, nms=False, opset=None,